##### Imports and setup

In [1]:
# setup the notebook environment
!pixi install
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import logging
logging.getLogger("pypsa.network.descriptors").setLevel(logging.ERROR)

def ensure_project_root_on_path(marker_dir: str = "modules") -> str:
    """Find the nearest ancestor folder containing `marker_dir`, add it to
    sys.path (front) and return its path. Raises RuntimeError if not found.
    """
    cwd = Path.cwd().resolve()
    # If cwd contains marker_dir, use cwd
    if (cwd / marker_dir).exists():
        p = str(cwd)
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    # Walk parents
    for parent in cwd.parents:
        if (parent / marker_dir).exists():
            p = str(parent)
            if p not in sys.path:
                sys.path.insert(0, p)
            return p
    raise RuntimeError(f"Could not find '{marker_dir}' directory in {cwd} or any parent")


project_root = ensure_project_root_on_path()
print("Added project root to sys.path:", project_root)

# imports and loading data
from modules.analysis_toolkit.helpers.config.constants import CAPTURE_RATE_IC
from modules.analysis_toolkit.helpers.index_finder import GeoOptions
from modules.analysis_toolkit.analyzer import ResultsComputer, GeoOptions, format_result
from modules.analysis_toolkit.helpers.plotting import TimeSeriesPlot, HistogramPlot, BarChartPlot, WaterfallPlot
import pandas as pd

study_years = [2030, 2040]
rc = {year: ResultsComputer(year=year, apply_snapshot_filter=False) for year in study_years}
weights = rc[2030].ns.get_iem_dispatch().snapshot_weightings["generators"]

✔ The default environment has been installed.
Added project root to sys.path: /Users/rca/PycharmProjects/NGV-FBMC


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Status Quo (SQ) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Integrated Energy Market (IEM) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Flow-based market coupling (FBMC) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'SQ (2030) - redispatch' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, sto

This notebook computes the key indicators reported in [Model run validation checks](https://docs.google.com/document/d/165EH2i8Fgtec260gfUZL6zXfYESi6Uwv6g7zTVdtgiU/edit?pli=1&tab=t.0) to check the consistency of the model runs and the results. The indicators are computed for both the dispatch and redispatch results, and can be compared to the values reported in FES25 or other reference documents.

# Dispatch

## Total GB demand
As reference, FES25 uses the following total GB demand:
* 339 TWh in 2030
* 564 TWh in 2040

In [2]:
pd.concat(
    [
        rc[year].withdrawal.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method withdrawal
Calling method withdrawal


year,2030,2040
scenario,,
sq,350.9,614.8
iem,350.8,614.9
iem_fb,350.8,616.3
diff: iem-sq,-0.1,0.0
diff: iemfb-iem,-0.0,1.4


## Total GB supply

In [3]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year,2030,2040
scenario,,
sq,403.2,695.3
iem,404.0,695.3
iem_fb,403.1,692.2
diff: iem-sq,0.8,0.0
diff: iemfb-iem,-1.0,-3.2


## Breakdown of GB generation per technology

In [4]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')")  \
            .groupby("carrier").sum() \
            .div(1e6) \
            .round(3)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year                        2030                                 \
scenario                      sq      iem   iem_fb diff: iem-sq   
carrier                                                           
Battery Storage            6.148    6.271    6.233        0.122   
Onshore Wind              69.478   69.614   69.623        0.136   
gas-ccgt                  29.080   29.499   28.578        0.419   
gas-ocgt                   0.000    0.000    0.000       -0.000   
geothermal                 0.063    0.063    0.063        0.000   
h2-ccgt                    0.000    0.000    0.000       -0.000   
home battery discharger    6.091    6.086    6.031       -0.005   
hydro-phs                  3.348    3.223    3.272       -0.125   
hydro-reservoir            4.689    4.695    4.695        0.006   
load                       0.000    0.000    0.000       -0.000   
nuclear                   20.017   20.017   20.017       -0.000   
offwind-dc-fl-oh         197.610  197.739  197.732        0.129   
solar-pv-utility          38.267   38.373   38.374        0.106   
solid biomass             28.444   28.437   28.433       -0.007   
waste                      0.000    0.000    0.000       -0.000   

year                                        2040                    \
scenario                diff: iemfb-iem       sq      iem   iem_fb   
carrier                                                              
Battery Storage                  -0.037    2.075    2.074    2.486   
Onshore Wind                      0.009   90.361   90.396   89.107   
gas-ccgt                         -0.921    5.893    5.745    5.829   
gas-ocgt                          0.000      NaN      NaN      NaN   
geothermal                        0.000    0.019    0.019    0.020   
h2-ccgt                           0.000    3.727    3.748    4.606   
home battery discharger          -0.055   76.957   77.076   74.936   
hydro-phs                         0.049    3.790    3.706    4.518   
hydro-reservoir                   0.000    4.677    4.694    4.632   
load                              0.000    0.000    0.000    0.000   
nuclear                           0.000   41.960   41.960   41.960   
offwind-dc-fl-oh                 -0.007  372.150  372.177  371.404   
solar-pv-utility                  0.001   81.838   81.859   80.890   
solid biomass                    -0.004   11.733   11.731   11.600   
waste                             0.000    0.145    0.163    0.163   

year                                                  
scenario                diff: iem-sq diff: iemfb-iem  
carrier                                               
Battery Storage               -0.002           0.412  
Onshore Wind                   0.036          -1.289  
gas-ccgt                      -0.147           0.084  
gas-ocgt                         NaN             NaN  
geothermal                    -0.000           0.002  
h2-ccgt                        0.021           0.859  
home battery discharger        0.119          -2.140  
hydro-phs                     -0.084           0.811  
hydro-reservoir                0.017          -0.062  
load                          -0.000           0.000  
nuclear                       -0.000           0.000  
offwind-dc-fl-oh               0.027          -0.773  
solar-pv-utility               0.020          -0.969  
solid biomass                 -0.002          -0.131  
waste                          0.018          -0.000

### Interconnector capacity

In [5]:
ic_capacity = pd.concat(
    [
        rc[year].ns.get_iem_dispatch().links.query(
            "carrier == 'DC' and (bus0.str.contains('GB ') or bus1.str.contains('GB ')) and ~name.str.contains('relation')"
        ) \
        .p_nom
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
ic_capacity.loc["Total", :] = ic_capacity.sum(axis=0)

In [6]:
ic_capacity

year,2030,2040
name,,
Aminth,1400.0,1400.0
BritNed,1200.0,1200.0
Continental Link,0.0,1800.0
Cronos,0.0,1400.0
East-West,585.0,585.0
ElecLink,1000.0,1000.0
FAB Link,0.0,1250.0
Greenlink (Greenwire),524.0,524.0
Gridlink,0.0,1250.0


## GB net position

In [7]:
pd.concat(
    [
        rc[year].energy_balance.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method energy_balance
Calling method energy_balance


year,2030,2040
scenario,,
sq,52.4,80.5
iem,53.2,80.5
iem_fb,52.3,75.9
diff: iem-sq,0.9,-0.0
diff: iemfb-iem,-0.9,-4.6


## GB prices

In [8]:
pd.concat(
    [
        rc[year].prices.compare_dispatch(bus_carrier=["AC", "AC_OH"]) \
            .query("name.str.contains('GB ')") \
            .describe() \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method prices
Calling method prices


year      2030                                            2040               \
scenario    sq   iem iem_fb diff: iem-sq diff: iemfb-iem    sq   iem iem_fb   
count     24.0  24.0   24.0         24.0            24.0  24.0  24.0   24.0   
mean      45.6  45.7   47.8          0.0             2.1  17.3  17.5   17.6   
std        2.9   2.7    3.6          0.7             2.2   1.1   1.2    1.3   
min       34.4  34.5   34.7         -1.7            -1.1  13.5  13.6   14.0   
25%       45.3  45.5   46.1          0.0             0.1  17.0  17.5   17.5   
50%       45.7  46.0   47.8          0.1             1.7  17.6  17.6   17.8   
75%       46.3  46.5   49.8          0.4             3.3  17.7  17.8   18.3   
max       52.0  50.3   52.6          1.3             7.1  18.8  19.7   20.1   

year                                   
scenario diff: iem-sq diff: iemfb-iem  
count            24.0            24.0  
mean              0.2             0.2  
std               0.6             1.0  
min              -1.0            -1.8  
25%              -0.1            -0.3  
50%               0.0             0.3  
75%               0.3             0.7  
max               2.3             2.4

## Congestion revenue

In [9]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum(axis=0) \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/133461534.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/133461534.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,4.3,-58.7
diff: iemfb-iem,-9.1,-0.2
iem,3803.2,3105.7
iem_fb,3794.1,3105.4
sq,3798.9,3164.4


### Breakdown of merchant congestion revenue per interconnector

In [10]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/429633954.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/429633954.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year                          2030                                       \
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq   
name                                                                      
Aminth                        -1.2            -0.1  436.5  436.4  437.7   
BritNed                        0.5            -0.8  492.0  491.2  491.5   
Continental Link               0.0             0.0    0.0    0.0    0.0   
Cronos                         0.0             0.0    0.0    0.0    0.0   
East-West                      0.5             0.1   51.9   51.9   51.3   
ElecLink                      -0.4            -2.5  342.0  339.5  342.3   
FAB Link                       0.0             0.0    0.0    0.0    0.0   
Greenlink (Greenwire)          0.5             0.1   46.4   46.6   46.0   
Gridlink                       0.0             0.0    0.0    0.0    0.0   
IFA                           -1.2            -2.5  683.9  681.4  685.1   
IFA2                          -0.4            -1.5  342.0  340.5  342.3   
Kulizumboo                     0.0             0.0    0.0    0.0    0.0   
LirIC                          0.0             0.0    0.0    0.0    0.0   
MARES                          0.7             0.1   66.5   66.6   65.8   
Moyle                          0.9            -0.0   88.6   88.6   87.7   
NS Link (NSL)                  6.0            -0.8  404.0  403.3  398.1   
Nautilus                       0.0             0.0    0.0    0.0    0.0   
Nemo                          -0.3            -1.2  413.0  411.8  413.4   
NeuConnect                     0.0             0.0    0.0    0.0    0.0   
NorthConnect                   0.0             0.0    0.0    0.0    0.0   
SENECA                         0.0             0.0    0.0    0.0    0.0   
Viking Link                   -1.2            -0.1  436.5  436.4  437.7   

year                          2040                                       
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq  
name                                                                     
Aminth                        -7.5            -0.0  103.6  103.5  111.1  
BritNed                       -1.9             1.1  187.9  189.0  189.9  
Continental Link              -8.8             4.5  145.3  149.8  154.1  
Cronos                        -2.4             0.6  224.2  224.8  226.6  
East-West                      0.6            -0.3   47.2   46.8   46.6  
ElecLink                      -1.4            -2.2  125.0  122.9  126.4  
FAB Link                      -1.8            -4.0  156.3  152.3  158.1  
Greenlink (Greenwire)          0.5            -0.2   42.3   42.0   41.7  
Gridlink                      -1.8             0.8  156.3  157.1  158.1  
IFA                           -2.8            -0.8  250.1  249.3  252.9  
IFA2                          -1.6            -2.5  125.0  122.5  126.6  
Kulizumboo                    -1.1            -3.4   87.5   84.2   88.6  
LirIC                          0.7            -0.4   56.4   56.1   55.8  
MARES                          0.7            -0.5   60.5   60.0   59.8  
Moyle                          1.0            -0.4   76.8   76.4   75.8  
NS Link (NSL)                 -7.0             3.6  119.5  123.1  126.5  
Nautilus                      -2.8             0.7  240.2  240.9  242.9  
Nemo                          -1.6            -3.2  160.1  156.9  161.8  
NeuConnect                    -3.1             3.9  328.5  332.4  331.5  
NorthConnect                  -6.8             1.1  113.0  114.1  119.9  
SENECA                        -2.0             1.3  196.4  197.7  198.4  
Viking Link                   -7.5            -0.0  103.6  103.5  111.1

## SEW change

In [11]:
pd.concat(
    [
        rc[year].welfare_system.compare_dispatch()
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
) # already in M€

Calling method welfare_system
Calling method welfare_system


year                   2030                                                   \
scenario                 sq       iem    iem_fb diff: iem-sq diff: iemfb-iem   
producer surplus   193632.2  193933.0  193919.7        300.8           -13.4   
consumer cost     -330767.2 -330902.4 -330918.5       -135.1           -16.1   
congestion income   21856.6   21876.4   21873.8         19.8            -2.6   
storage surplus     38892.1   38719.6   38742.7       -172.5            23.1   
total_welfare      -76386.3  -76373.4  -76382.4         12.9            -9.0   

year                   2040                                                   
scenario                 sq       iem    iem_fb diff: iem-sq diff: iemfb-iem  
producer surplus    97405.6   97945.8   99089.9        540.2          1144.1  
consumer cost     -201214.7 -202005.2 -203310.0       -790.5         -1304.8  
congestion income   27546.4   27572.2   27563.4         25.8            -8.8  
storage surplus     22224.9   22452.7   22577.2        227.8           124.5  
total_welfare      -54037.8  -54034.5  -54079.5          3.3           -45.0

# Redispatch

## Constraint costs

In [12]:
pd.concat(
    [
        rc[year].constraint_costs.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1)
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_costs
Calling method constraint_costs


year,2030,2040
scenario,,
diff: iem-sq,38.5,19.4
diff: iemfb-iem,3.9,-205.2
iem,391.1,6349.9
iem_fb,395.0,6144.8
sq,352.7,6330.6


## Total redispatched energy volume
redispatch + countertrading + load shedding

In [13]:
pd.concat(
    [
        rc[year].constraint_management_volume.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_management_volume
Calling method constraint_management_volume


year,2030,2040
scenario,,
diff: iem-sq,-1.9,-1.4
diff: iemfb-iem,-2.5,0.5
iem,313.7,407.9
iem_fb,311.1,408.3
sq,315.6,409.3


### Redispatch

#### Redispatch: ramp up

In [14]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()


Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/1384244134.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/1384244134.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-0.4,-0.1
diff: iemfb-iem,-0.7,-0.7
iem,136.4,157.9
iem_fb,135.8,157.2
sq,136.9,158.0


#### Redispatch: ramp down

In [15]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()

Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/4250467477.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/4250467477.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,0.2,-0.1
diff: iemfb-iem,-1.5,-1.3
iem,151.2,193.7
iem_fb,149.7,192.4
sq,151.0,193.8


### Countertrading

#### Countertrading: ramp up

In [16]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/999498334.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/999498334.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-0.6,-0.7
diff: iemfb-iem,-0.6,0.6
iem,20.4,45.7
iem_fb,19.8,46.3
sq,21.0,46.4


#### Countertrading: ramp down

In [17]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/983782068.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/983782068.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-1.2,-0.5
diff: iemfb-iem,0.2,1.8
iem,5.6,10.6
iem_fb,5.9,12.4
sq,6.8,11.1


### Load shedding (ramp up only)

In [18]:
pd.concat(
    [
        rc[year].load_shedding_volume.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method load_shedding_volume
Calling method load_shedding_volume


year,2030,2040
scenario,,
diff: iem-sq,0.0,0.0
diff: iemfb-iem,0.0,0.0
iem,0.0,0.0
iem_fb,0.0,0.0
sq,0.0,0.0


## Redispatch cost per MWh, ignoring load shedding

In [19]:
pd.concat(
    [
        rc[year].average_ramping_costs_per_mw.compare_redispatch()
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method average_ramping_costs_per_mw
Calling method average_ramping_costs_per_mw


year                2030                                            2040  \
scenario              sq   iem iem_fb diff: iem-sq diff: iemfb-iem    sq   
avg_cost_ramp_up    45.3  46.1   45.6          0.8            -0.5  15.5   
avg_cost_ramp_down -43.1 -43.6  -43.1         -0.5             0.5  15.5   

year                                                          
scenario             iem iem_fb diff: iem-sq diff: iemfb-iem  
avg_cost_ramp_up    14.9   14.7         -0.6            -0.1  
avg_cost_ramp_down  16.3   15.3          0.8            -0.9

# PLAYGROUND

In [20]:
# redispatch opex
rc[2040].opex.compare_redispatch().query("carrier.str.contains('ramp')").sum().div(1e6).round(1)

Calling method opex


scenario
sq                 5634.5
iem                5654.8
iem_fb             5486.3
diff: iem-sq         20.4
diff: iemfb-iem    -168.5
dtype: float64

In [21]:
# net boundary loading (-1, 1, both directions) statistics via describe()
format_result(
    result=rc[2030].boundary_loading.compare_redispatch(which="actual").xs("DIRECT", level="direction").sub(
        rc[2030].boundary_loading.compare_redispatch(which="actual").xs("OPPOSITE", level="direction")
    ),
    sum_by=["scenario", "snapshot", "boundary"],
    factor=1
).unstack(["boundary"]).describe().stack("boundary").sort_index(level="boundary").swaplevel(-1, 0).round(3)

Calling method boundary_loading
Calling method boundary_loading
Grouping by ['scenario', 'snapshot', 'boundary'] aggregates across [].


/Users/rca/PycharmProjects/NGV-FBMC/modules/analysis_toolkit/analyzer.py:27: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.

/Users/rca/PycharmProjects/NGV-FBMC/modules/analysis_toolkit/analyzer.py:28: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_23410/3249623454.py:8: FutureWarning:

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.



scenario        diff: iem-sq  diff: iemfb-iem       iem    iem_fb        sq
boundary                                                                   
B0       25%          -0.001           -0.000     0.099     0.099     0.100
         50%          -0.000            0.000     0.263     0.263     0.263
         75%           0.001            0.001     0.455     0.455     0.458
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.049            0.024     0.731     0.730     0.727
...                      ...              ...       ...       ...       ...
SW1      count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.421            0.358     0.730     0.732     0.729
         mean          0.010            0.000     0.202     0.203     0.192
         min          -0.412           -0.338    -0.480    -0.480    -0.472
         std           0.080            0.042     0.202     0.202     0.203

[176 rows x 5 columns]

In [22]:
year = 2030
shedding_volumes = rc[year].load_shedding_volume.compare_redispatch()
shedding_snapshots = shedding_volumes.loc[:, shedding_volumes.sum(axis=0)>0].columns.get_level_values("snapshot")

Calling method load_shedding_volume


In [23]:
shedding_snapshots

DatetimeIndex([], dtype='datetime64[ns]', name='snapshot', freq=None)

In [24]:
available_redispatch_capacity = rc[2030].available_redispatch_capacity_mw.sq_redispatch().filter(regex='^(?!.*EU)', axis=1)

Calling method available_redispatch_capacity_mw


In [25]:
available_redispatch_capacity.loc[shedding_snapshots].filter(like="ramp up").sort_values(by=shedding_snapshots[0], axis=1, ascending=False)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
used_redispatch_capacity = rc[2030].used_redispatch_capacity_pu.sq_redispatch().filter(regex='^(?!.*EU)', axis=1)

In [ ]:
used_redispatch_capacity.loc[shedding_snapshots[0]].filter(like="ramp up").sort_values().head(20).round(2)#(by=shedding_snapshots[0], axis=1, ascending=True)

In [ ]:
net_position_gb_buses = rc[2030].net_position.compare_dispatch().groupby("bus").sum().xs(shedding_snapshots[0], level="snapshot", axis=1)

In [ ]:
net_position_gb_buses.filter(like="GB ", axis=0).round()